In [2]:
import requests
import json
from datetime import datetime, timezone

# 설정 정보 (실제 환경에 맞게 변경 필요)
AIRFLOW_HOST = "http://10.99.212.69:8080" # Airflow Webserver 주소
AIRFLOW_USER = "airflow"                   # Airflow API 사용자 ID
AIRFLOW_PASS = "airflow"               # Airflow API 비밀번호

DAG_ID = "subscription_trigger"
URL = f"{AIRFLOW_HOST}/api/v1/dags/{DAG_ID}/dagRuns"

def get_jwt_token(host, user, password):
    token_url = f"{host}/auth/token"
    payload = {"username": user, "password": password}
    
    try:
        response = requests.post(token_url, json=payload)
        response.raise_for_status()
        return response.json().get('access_token')
    except requests.exceptions.HTTPError as e:
        print(f"토큰 획득 실패 (HTTP Error): {e}")
        print(f"응답 내용: {response.text}")
        return None

In [3]:
# 2. DAG 트리거 함수
def trigger_airflow_dag_jwt(cell_id_input):
    
    # 1. 토큰 획득
    jwt_token = get_jwt_token(AIRFLOW_HOST, AIRFLOW_USER, AIRFLOW_PASS)
    if not jwt_token:
        return False
    
    # API v2 사용 및 엔드포인트 수정 (dag_runs 언더바 사용을 권장)
    # 또한, 405 오류를 피하기 위해 엔드포인트를 `/dag_runs`로 시도해봅니다.
    trigger_url = f"{AIRFLOW_HOST}/api/v2/dags/{DAG_ID}/dagRuns" 
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {jwt_token}" # 획득한 토큰을 헤더에 추가
    }
    
    payload = {
        "conf": {"cell_id": cell_id_input},
        "logical_date": datetime.now(timezone.utc).isoformat()
    }
    
    try:
        response = requests.post(
            trigger_url,
            headers=headers,
            data=json.dumps(payload)
        )
        response.raise_for_status()
        
        print(f"DAG 트리거 성공! DAG Run ID: {response.json().get('dag_run_id')}")
        return True
        
    except requests.exceptions.HTTPError as e:
        print(f"DAG 트리거 실패 (HTTP Error): {e}")
        print(f"응답 내용: {response.text}")
        return False

# 예시 사용:
success = trigger_airflow_dag_jwt("DA05B19225")

DAG 트리거 성공! DAG Run ID: manual__2025-10-29T05:13:31.373209+00:00
